In [ ]:
# Cell 1: Setup - imports, auth, paths and table order
import pandas as pd
import os
import numpy as np
import math
from google.colab import drive
from google.colab import userdata
from supabase import create_client

# Mount drive (Colab) - skip if already mounted
drive.mount('/content/drive')

SUPABASE_URL = "https://ysbchbtuubrphiulkvlh.supabase.co"
SUPABASE_KEY = userdata.get('API_KEY')
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
CSV_PATH = "/content/drive/MyDrive/CSV"

print('Setup completo. Cambia `CSV_PATH` o `SUPABASE_KEY` si hace falta.')

# Orden por defecto; puedes ejecutar tablas individualmente cambiando esta lista
tables_order = [
    ('geolocation', 'olist_geolocation_dataset.csv'),
    ('product_category_name_translation', 'product_category_name_translation.csv'),
    ('customers', 'olist_customers_dataset.csv'),
    ('sellers', 'olist_sellers_dataset.csv'),
    ('products', 'olist_products_dataset.csv'),
    ('orders', 'olist_orders_dataset.csv'),
    ('order_items', 'olist_order_items_dataset.csv'),
    ('order_payments', 'olist_order_payments_dataset.csv'),
    ('order_reviews', 'olist_order_reviews_dataset.csv')
]


In [ ]:
# Cell 2: Helpers - cleaning, sanitization and chunking

def clean_dates_in_df(df):
    for col in df.columns:
        if 'date' in col or 'timestamp' in col:
            df[col] = pd.to_datetime(df[col], errors='coerce').dt.strftime('%Y-%m-%dT%H:%M:%S')
    return df


def _sanitize_value(v):
    if v is None:
        return None
    # numpy integer
    if isinstance(v, (np.integer,)):
        return int(v)
    # numpy floating
    if isinstance(v, (np.floating,)):
        f = float(v)
        if math.isnan(f) or math.isinf(f):
            return None
        if f.is_integer():
            return int(f)
        return f
    # native float
    if isinstance(v, float):
        if math.isnan(v) or math.isinf(v):
            return None
        if v.is_integer():
            return int(v)
        return v
    return v


def sanitize_record(rec):
    return {k: _sanitize_value(v) for k, v in rec.items()}


def chunked(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i+size]


In [ ]:
# Cell 3: Prepare data and convert to sanitized records

def df_to_sanitized_records(df, table_name, categorias_validas=None):
    # replace inf with nan
    df = df.replace([np.inf, -np.inf], np.nan)

    # fk validation example for products
    if table_name == 'products' and categorias_validas is not None:
        df['product_category_name'] = df['product_category_name'].apply(lambda x: x if x in categorias_validas else None)

    df = clean_dates_in_df(df)
    raw = df.to_dict(orient='records')
    return [sanitize_record(r) for r in raw]


def read_csv_prepare(path):
    df = pd.read_csv(path)
    return df


In [ ]:
# Cell 4: Upload utilities (per-table, batch uploader, dry-run)

def upload_records(table_name, records, batch_size=500, dry_run=True):
    """Upload sanitized records in batches. If dry_run=True, prints first batch and returns."""
    if len(records) == 0:
        print('Sin registros para subir.')
        return
    if dry_run:
        print(f"Dry-run: primer lote ({min(batch_size, len(records))} registros) para tabla {table_name}:")
        from pprint import pprint
        pprint(records[:batch_size])
        return

    for i, batch in enumerate(chunked(records, batch_size)):
        try:
            supabase.table(table_name).insert(batch).execute()
            print(f"Lote {i*batch_size}.. subido correctamente ({len(batch)} registros)")
        except Exception as e:
            print(f"Error en lote {i*batch_size} de {table_name}: {e}")


def upload_table_from_csv(table_name, file_name, categorias_validas=None, sample_rows=None, dry_run=True, batch_size=500):
    file_path = os.path.join(CSV_PATH, file_name)
    if not os.path.exists(file_path):
        print('Archivo no encontrado:', file_path)
        return

    df = read_csv_prepare(file_path)
    if sample_rows is not None:
        df = df.head(sample_rows)

    records = df_to_sanitized_records(df, table_name, categorias_validas=categorias_validas)
    upload_records(table_name, records, batch_size=batch_size, dry_run=dry_run)


In [ ]:
# Cell 5: Quick dry-run example for `products` (prints sanitized batch, no upload)

# Cambia sample_rows a 1000 para ver más ejemplos
upload_table_from_csv('products', 'olist_products_dataset.csv', categorias_validas=None, sample_rows=200, dry_run=True, batch_size=100)

# Si quieres probar con categorias válidas, ejecuta:
# df_cat = pd.read_csv(os.path.join(CSV_PATH, 'product_category_name_translation.csv'))
# categorias_validas = df_cat['product_category_name'].unique()
# upload_table_from_csv('products', 'olist_products_dataset.csv', categorias_validas=categorias_validas, sample_rows=200, dry_run=True)


In [ ]:
# Cell 6: Diagnostics for `order_reviews` duplicates (within CSV)

# Detectar IDs duplicados dentro del CSV (antes de subir)
file_path = os.path.join(CSV_PATH, 'olist_order_reviews_dataset.csv')
if os.path.exists(file_path):
    df_reviews = pd.read_csv(file_path)
    dup_ids = df_reviews[df_reviews.duplicated(subset=['review_id'], keep=False)]['review_id']
    if dup_ids.empty:
        print('No hay `review_id` duplicados dentro del CSV.')
    else:
        print('Ejemplos de review_id duplicados (primos 20):')
        print(dup_ids.drop_duplicates().head(20).to_list())
        # muestra las filas duplicadas para el primer id duplicado
        first = dup_ids.drop_duplicates().iloc[0]
        print('\nFilas duplicadas para el primer review_id:')
        print(df_reviews[df_reviews['review_id'] == first])
else:
    print('Archivo order_reviews no encontrado:', file_path)

# Nota: si los duplicados vienen de cargas anteriores en la BD, ejecuta consultas SELECT para ver existencias en la tabla remota antes de subir.


In [ ]:
# Cell 7: Example small upload (uncomment to run) — use cautiously

# Ejemplo: subir 1000 filas de products (descomenta para ejecutar)
# categorias_validas = pd.read_csv(os.path.join(CSV_PATH, 'product_category_name_translation.csv'))['product_category_name'].unique()
# upload_table_from_csv('products', 'olist_products_dataset.csv', categorias_validas=categorias_validas, sample_rows=1000, dry_run=False, batch_size=200)

# Ejemplo: probar sólo order_reviews dry-run
# upload_table_from_csv('order_reviews', 'olist_order_reviews_dataset.csv', sample_rows=500, dry_run=True)

# Full run (descomenta sólo cuando estés listo)
# for tname, fname in tables_order:
#     upload_table_from_csv(tname, fname, categorias_validas=None, sample_rows=None, dry_run=False, batch_size=500)


In [ ]:
# Cell X: Diagnostic - comprueba existencia de CSVs y muestra primeras filas
import glob

print('CSV_PATH =', CSV_PATH)
if not os.path.exists(CSV_PATH):
    print('Ruta no encontrada:', CSV_PATH)
else:
    files = sorted(glob.glob(os.path.join(CSV_PATH, '*.csv')))
    print(f'CSV encontrados ({len(files)}):')
    for f in files:
        print('-', os.path.basename(f))

    # Mostrar primeras filas de cada archivo relevante
    check_files = [
        'olist_geolocation_dataset.csv',
        'product_category_name_translation.csv',
        'olist_customers_dataset.csv',
        'olist_sellers_dataset.csv',
        'olist_products_dataset.csv',
        'olist_orders_dataset.csv',
        'olist_order_items_dataset.csv',
        'olist_order_payments_dataset.csv',
        'olist_order_reviews_dataset.csv'
    ]

    for name in check_files:
        path = os.path.join(CSV_PATH, name)
        if os.path.exists(path):
            try:
                print('\n---', name, '---')
                df_tmp = pd.read_csv(path, nrows=5)
                print(df_tmp.head().to_string())
            except Exception as e:
                print('Error leyendo', name, e)
        else:
            print('\nNo existe:', name)
